# Atelier pandas : de vos tableaux Excel à un tableau de bord

Bienvenue ! Ici, **rien à installer** : tout se passe dans votre navigateur, et vos données ne quittent pas votre ordinateur.

**Comment ça marche ?**
1. Cliquez sur une case grise (une « cellule de code »).
2. Appuyez sur **Maj + Entrée** (ou sur le bouton ▶ en haut). Le résultat s'affiche dessous.
3. Passez à la case suivante et recommencez.

La première exécution est un peu longue (une trentaine de secondes) : le navigateur prépare Python. C'est normal.

> Vous ne pouvez rien casser. En cas de souci : menu **Noyau (Kernel) → Redémarrer**, puis reprenez depuis le début.

## 1. Charger un tableau
On charge un fichier CSV, l'équivalent d'un Excel « à plat ». Ce tableau s'appelle un **DataFrame** (souvent abrégé `df`).

In [ ]:
import pandas as pd

df = pd.read_csv("donnees/batiments_exemple.csv", sep=";")
df.head()      # affiche les 5 premières lignes

*Les données de cet atelier sont fictives, dans le style des données ouvertes de la Ville. Pour utiliser un vrai fichier, déposez-le dans le dossier `donnees` (panneau de gauche) et changez le nom entre guillemets.*

## 2. Regarder ce qu'on a

In [ ]:
print("Nombre de lignes et de colonnes :", df.shape)
df.info()      # colonnes, types, valeurs manquantes

In [ ]:
df.describe()   # statistiques rapides : moyenne, min, max...

## 3. Nettoyer
Sur la colonne `volume_m3`, quelques cases sont vides. On les repère, puis on retire ces lignes.

In [ ]:
print("Cases vides dans volume_m3 :", df["volume_m3"].isna().sum())
df = df.dropna(subset=["volume_m3"])
print("Il reste", len(df), "lignes")

## 4. Filtrer
Comme le filtre d'Excel : on garde seulement les lignes qui répondent à une condition.

In [ ]:
bureaux = df[df["type_batiment"] == "Bureau"]
bureaux.head()

**À vous de jouer :** remplacez `"Bureau"` par `"Commercial"`, puis ré-exécutez la case (Maj + Entrée).

## 5. Regrouper (« regrouper par »)
Combien de m³ par arrondissement ? C'est le principe du sous-total.

In [ ]:
volumes = df.groupby("arrondissement")["volume_m3"].sum().sort_values(ascending=False)
volumes.head(5)     # les 5 plus gros

## 6. Le tableau croisé dynamique (TCD)
Exactement ce que vous faites dans Excel en glissant des champs dans *Lignes*, *Colonnes* et *Valeurs*.

In [ ]:
tcd = pd.pivot_table(
    df,
    index="arrondissement",      # les lignes
    columns="type_batiment",     # les colonnes
    values="volume_m3",          # ce qu'on additionne
    aggfunc="sum",               # somme (essayez "mean" pour la moyenne)
    fill_value=0,                # 0 à la place des cases vides
    margins=True,                # ajoute les totaux
    margins_name="Total",
)
tcd

## 7. Un graphique
Une ligne suffit.

In [ ]:
import matplotlib.pyplot as plt

tcd.drop(index="Total", columns="Total").plot(kind="bar", stacked=True, figsize=(11, 5),
                                               title="Volumes bâtis par arrondissement")
plt.ylabel("m³")
plt.show()

**À vous de jouer :** dans la case du TCD, remplacez `"type_batiment"` par `"annee"`, ré-exécutez le TCD, puis le graphique.

## 8. Récupérer votre résultat
On enregistre le TCD en Excel, puis un lien de téléchargement apparaît.

In [ ]:
try:                                # spécifique au navigateur : charge l'outil Excel
    import piplite
    await piplite.install("openpyxl")
except ImportError:
    pass
import openpyxl                    # nécessaire pour écrire un fichier Excel
import base64
from IPython.display import HTML, display

def telecharger(nom, ouvrir=False):
    """Affiche un lien pour enregistrer le fichier sur votre ordinateur
    (et, si ouvrir=True, un second lien pour l'ouvrir dans un nouvel onglet)."""
    contenu = open(nom, "rb").read()
    try:                                    # dans le navigateur
        import js
        from pyodide.ffi import to_js
        blob = js.Blob.new(to_js([to_js(contenu)]),
                           to_js({"type": "application/octet-stream"}, dict_converter=js.Object.fromEntries))
        adresse = js.URL.createObjectURL(blob)
    except ImportError:                     # Python classique (hors navigateur)
        adresse = "data:application/octet-stream;base64," + base64.b64encode(contenu).decode()
    liens = (f'<a download="{nom}" href="{adresse}" style="font-size:1.1em">'
             f'⬇ Cliquez ici pour télécharger {nom}</a>')
    if ouvrir:
        liens += (f' &nbsp;&nbsp; <a target="_blank" href="{adresse}" style="font-size:1.1em">'
                  f'Ouvrir dans un nouvel onglet</a>')
    display(HTML(liens))

tcd.to_excel("mon_tcd.xlsx", sheet_name="TCD")
telecharger("mon_tcd.xlsx")

Si le lien ne fonctionne pas : dans le panneau de gauche, clic droit sur `mon_tcd.xlsx` puis **Télécharger**.

**Bravo, vous venez de refaire un TCD Excel avec du code.**

## 9. Des graphiques interactifs (Plotly)
Les graphiques précédents sont des images. Avec **Plotly**, on peut survoler, zoomer et filtrer avec la souris.

D'abord on charge Plotly (quelques secondes, uniquement la première fois).

In [ ]:
try:                                # spécifique au navigateur
    import piplite
    await piplite.install("plotly")
except ImportError:
    pass

import plotly.express as px
print("Plotly est prêt")

In [ ]:
ordre = ["1er"] + [f"{i}e" for i in range(2, 21)]

fig_arr = px.bar(
    df.groupby(["arrondissement", "type_batiment"], as_index=False)["volume_m3"].sum(),
    x="arrondissement", y="volume_m3", color="type_batiment",
    category_orders={"arrondissement": ordre},
    title="Volumes bâtis par arrondissement et par type",
)

fig_annee = px.line(
    df.groupby("annee", as_index=False)["volume_m3"].sum(),
    x="annee", y="volume_m3", markers=True,
    title="Évolution des volumes par année",
)

fig_type = px.pie(df, names="type_batiment", values="volume_m3",
                  title="Répartition des volumes par type de bâtiment")

**À vous de jouer :** dans `fig_arr`, remplacez `px.bar` par `px.line` (ou `px.area`) et ré-exécutez.

## 10. Télécharger votre tableau de bord (une page web autonome)
La cellule ci-dessous assemble vos graphiques dans **une seule page web** : le texte (HTML), la mise en forme (CSS) et le moteur des graphiques (JavaScript Plotly) sont tous dans le même fichier.

Vous pouvez l'envoyer par mail ou le double-cliquer : il s'ouvre dans n'importe quel navigateur, **sans internet et sans Python**.

In [ ]:
import html
from plotly.offline import get_plotlyjs

STYLE = """
:root { --papier:#EEF2F5; --encre:#142434; --trait:#C3CED8; --bleu:#1F5A99; --piquet:#E0A81C; }
* { box-sizing: border-box; }
body { margin:0; background:var(--papier); color:var(--encre); font:16px/1.5 "Segoe UI", Arial, sans-serif; }
header { padding:32px 5vw 20px; border-bottom:1px solid var(--trait); border-left:8px solid var(--piquet); }
h1 { margin:0; font:600 2rem/1.1 "Bahnschrift","Segoe UI",Arial,sans-serif; }
main { display:grid; grid-template-columns:repeat(auto-fit,minmax(min(100%,520px),1fr)); gap:24px; padding:24px 5vw 48px; }
.planche { background:#fff; border:1px solid var(--trait); padding:8px; min-width:0; }
"""

def exporter_dashboard(graphiques, titre="Mon tableau de bord", fichier="mon_dashboard.html"):
    """graphiques = liste de graphiques Plotly. Écrit une page web autonome."""
    blocs = "\n".join(
        '<section class="planche">' + g.to_html(full_html=False, include_plotlyjs=False,
                                                config={"responsive": True}) + "</section>"
        for g in graphiques)
    page = (f'<!DOCTYPE html><html lang="fr"><head><meta charset="utf-8">'
            f'<meta name="viewport" content="width=device-width, initial-scale=1">'
            f'<title>{html.escape(titre)}</title><style>{STYLE}</style>'
            f'<script>{get_plotlyjs()}</script></head>'
            f'<body><header><h1>{html.escape(titre)}</h1></header><main>{blocs}</main></body></html>')
    with open(fichier, "w", encoding="utf-8") as f:
        f.write(page)
    return fichier

fichier = exporter_dashboard([fig_arr, fig_annee, fig_type], titre="Volumes bâtis (atelier pandas)")
telecharger(fichier, ouvrir=True)

Un peu lent ? C'est normal : la page contient tout le moteur des graphiques (environ 5 Mo).

Si le lien ne réagit pas : panneau de gauche, clic droit sur `mon_dashboard.html` puis **Télécharger**.

**Vous avez maintenant un vrai tableau de bord web, créé par vos soins.**